# Data Lake Assignment — Bronze and Silver Layers

This notebook implements the required **Bronze → Silver** workflow using **Apache Spark / PySpark**.

### Goals
- Keep the supplied `sales.csv` unchanged in the Bronze layer.
- Inspect the data with Spark and measure discrepancies dynamically.
- Perform only basic preprocessing needed for the Silver layer.
- Preserve useful records wherever possible.
- Save the cleaned result as `sales_silver.csv`.

> **Submission note:** The class repository already contains the original dataset and README. According to the submission instructions, only the notebook and Silver CSV should be added inside your own `submissions/<ROLL_NUMBER>_<NAME>/` folder.

## 1. Install PySpark and create the Data Lake folders

The notebook creates the required runtime structure:

```text
data_lake/
├── bronze/
│   └── sales.csv
└── silver/
    └── preprocessed_sales/
```

In [1]:
# Install the Spark package used in Google Colab
!pip -q install pyspark

from pathlib import Path
import hashlib
import shutil
import os

from google.colab import files
from pyspark.sql import SparkSession, functions as F

BASE = Path("data_lake")
BRONZE_DIR = BASE / "bronze"
SILVER_DIR = BASE / "silver" / "preprocessed_sales"
BRONZE_FILE = BRONZE_DIR / "sales.csv"
SILVER_FILE = SILVER_DIR / "sales_silver.csv"
SUBMISSION_SILVER = Path("silver") / "sales_silver.csv"

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_SILVER.parent.mkdir(parents=True, exist_ok=True)

print("Data Lake folders are ready.")
print(f"Bronze path: {BRONZE_DIR}")
print(f"Silver path: {SILVER_DIR}")

Data Lake folders are ready.
Bronze path: data_lake/bronze
Silver path: data_lake/silver/preprocessed_sales


## 2. Upload the raw dataset into Bronze

Upload the **original `sales.csv` supplied with the assignment**. The file is copied byte-for-byte into Bronze; it is not opened, cleaned, renamed internally, or rewritten before Spark reads it.

In [2]:
uploaded = files.upload()

if "sales.csv" not in uploaded:
    raise FileNotFoundError(
        "Please upload the original file with the exact filename 'sales.csv'."
    )

# Copy the uploaded bytes directly into Bronze.
shutil.copyfile("sales.csv", BRONZE_FILE)


def sha256sum(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_hash = sha256sum(Path("sales.csv"))
bronze_hash = sha256sum(BRONZE_FILE)

print(f"Bronze file created: {BRONZE_FILE}")
print(f"SHA-256 of uploaded file: {source_hash}")
print(f"SHA-256 of Bronze file:   {bronze_hash}")
print("Bronze unchanged:", source_hash == bronze_hash)

assert source_hash == bronze_hash, "Bronze file is not byte-for-byte identical to the uploaded source."


Saving sales.csv to sales.csv
Bronze file created: data_lake/bronze/sales.csv
SHA-256 of uploaded file: 1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8
SHA-256 of Bronze file:   1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8
Bronze unchanged: True


In [3]:
# Show the required directory structure
for root, dirs, filenames in os.walk(BASE):
    level = Path(root).relative_to(BASE).parts
    indent = "  " * len(level)
    print(f"{indent}{Path(root).name}/")
    for name in sorted(filenames):
        print(f"{indent}  {name}")

data_lake/
  bronze/
    sales.csv
  silver/
    preprocessed_sales/


## 3. Start Spark and load the Bronze data

The Bronze CSV is read with Spark schema inference so the inspection uses Spark DataFrame operations.

In [4]:
spark = (
    SparkSession.builder
    .appName("SalesDataLakeBronzeToSilver")
    .getOrCreate()
)

bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(str(BRONZE_FILE))
)

print("=== Bronze Schema ===")
bronze_df.printSchema()

print("=== First 5 Rows ===")
bronze_df.show(5, truncate=False)

bronze_count = bronze_df.count()
print(f"Total Bronze records: {bronze_count}")

=== Bronze Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

=== First 5 Rows ===
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------

## 4. Identify duplicates and missing values

All discrepancy counts below are calculated from Spark; no counts are hard-coded.

In [5]:
# Exact duplicate rows
unique_rows = bronze_df.dropDuplicates().count()
duplicate_rows = bronze_count - unique_rows

print(f"Exact duplicate rows: {duplicate_rows}")

# Null / blank values by column
missing_exprs = []
for column_name, data_type in bronze_df.dtypes:
    if data_type == "string":
        condition = F.col(column_name).isNull() | (F.length(F.trim(F.col(column_name))) == 0)
    else:
        condition = F.col(column_name).isNull()
    missing_exprs.append(F.sum(F.when(condition, 1).otherwise(0)).alias(column_name))

print("\n=== Missing / Blank Values ===")
bronze_df.select(missing_exprs).show(truncate=False)

Exact duplicate rows: 10

=== Missing / Blank Values ===
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



## 5. Inspect text-format inconsistencies

The checks below look for multiple raw spellings that become the same value after trimming and lower-casing. This is useful for spotting obvious capitalization/spacing variants without over-cleaning the data.

In [6]:
def show_text_variants(frame, column_name):
    variants = (
        frame
        .filter(F.col(column_name).isNotNull())
        .withColumn("normalized_value", F.lower(F.trim(F.col(column_name))))
        .groupBy("normalized_value")
        .agg(
            F.countDistinct(F.col(column_name)).alias("different_raw_forms"),
            F.sort_array(F.collect_set(F.col(column_name))).alias("raw_forms")
        )
        .filter(F.col("different_raw_forms") > 1)
        .orderBy(F.desc("different_raw_forms"), F.col("normalized_value"))
    )
    print(f"\nText variants detected in: {column_name}")
    variants.show(truncate=False)

show_text_variants(bronze_df, "category")
show_text_variants(bronze_df, "city")

print("\nCategory values as stored in Bronze:")
bronze_df.groupBy("category").count().orderBy("category").show(truncate=False)


Text variants detected in: category
+----------------+-------------------+--------------------------+
|normalized_value|different_raw_forms|raw_forms                 |
+----------------+-------------------+--------------------------+
|sports          |3                  |[SPORTS, Sports, sports]  |
|beauty          |2                  |[Beauty, beauty]          |
|clothing        |2                  |[CLOTHING, Clothing]      |
|electronics     |2                  |[ELECTRONICS, Electronics]|
+----------------+-------------------+--------------------------+


Text variants detected in: city
+----------------+-------------------+-----------------------------------+
|normalized_value|different_raw_forms|raw_forms                          |
+----------------+-------------------+-----------------------------------+
|delhi           |3                  |[DELHI, Delhi, delhi]              |
|hyderabad       |3                  |[ Hyderabad , HYDERABAD, Hyderabad]|
|coimbatore      |2       

## 6. Inspect numerical values

For the three required numerical fields, values that violate simple business rules are measured with Spark:
- `quantity` should be greater than 0.
- `unit_price` should be greater than 0.
- `discount_percent` should be between 0 and 100 inclusive.

The inspection also checks whether any values fail numeric conversion.

In [7]:
numeric_rules = {
    "quantity": lambda x: x <= 0,
    "unit_price": lambda x: x <= 0,
    "discount_percent": lambda x: (x < 0) | (x > 100),
}

numeric_checks = []
for column_name, invalid_rule in numeric_rules.items():
    numeric_value = F.col(column_name).cast("double")
    numeric_checks.extend([
        F.sum(
            F.when(
                F.col(column_name).isNotNull() & numeric_value.isNull(), 1
            ).otherwise(0)
        ).alias(f"{column_name}_non_numeric"),
        F.sum(
            F.when(numeric_value.isNotNull() & invalid_rule(numeric_value), 1)
             .otherwise(0)
        ).alias(f"{column_name}_rule_violations")
    ])

print("=== Numerical Validation Checks ===")
bronze_df.select(numeric_checks).show(truncate=False)

print("=== Numerical Ranges ===")
bronze_df.select(
    F.min(F.col("quantity").cast("double")).alias("min_quantity"),
    F.max(F.col("quantity").cast("double")).alias("max_quantity"),
    F.min(F.col("unit_price").cast("double")).alias("min_unit_price"),
    F.max(F.col("unit_price").cast("double")).alias("max_unit_price"),
    F.min(F.col("discount_percent").cast("double")).alias("min_discount_percent"),
    F.max(F.col("discount_percent").cast("double")).alias("max_discount_percent")
).show()

=== Numerical Validation Checks ===
+--------------------+------------------------+----------------------+--------------------------+----------------------------+--------------------------------+
|quantity_non_numeric|quantity_rule_violations|unit_price_non_numeric|unit_price_rule_violations|discount_percent_non_numeric|discount_percent_rule_violations|
+--------------------+------------------------+----------------------+--------------------------+----------------------------+--------------------------------+
|0                   |12                      |0                     |8                         |0                           |8                               |
+--------------------+------------------------+----------------------+--------------------------+----------------------------+--------------------------------+

=== Numerical Ranges ===
+------------+------------+--------------+--------------+--------------------+--------------------+
|min_quantity|max_quantity|min_unit_pr

## 7. Inspect inconsistent or invalid dates

The dataset contains more than one reasonable date spelling, so several known patterns are tried. Anything that cannot be parsed by those formats is treated as invalid rather than being guessed.

In [8]:
date_formats = [
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd/MM/yyyy",
    "MM-dd-yyyy",
    "yyyy-dd-MM",
]

trimmed_date = F.trim(F.col("order_date"))
parsed_date = F.coalesce(*[
    F.try_to_timestamp(trimmed_date, F.lit(pattern))
    for pattern in date_formats
])

missing_dates = bronze_df.filter(
    F.col("order_date").isNull() | (F.length(trimmed_date) == 0)
).count()

invalid_dates = bronze_df.filter(
    F.col("order_date").isNotNull()
    & (F.length(trimmed_date) > 0)
    & parsed_date.isNull()
).count()

print(f"Missing/blank dates: {missing_dates}")
print(f"Invalid dates not matching supported formats: {invalid_dates}")

print("\nExamples of date values that need special handling:")
bronze_df.select("order_date").distinct().orderBy("order_date").show(20, truncate=False)

Missing/blank dates: 0
Invalid dates not matching supported formats: 3

Examples of date values that need special handling:
+----------+
|order_date|
+----------+
|09-15-2026|
|2025-01-01|
|2025-01-02|
|2025-01-03|
|2025-01-04|
|2025-01-06|
|2025-01-08|
|2025-01-09|
|2025-01-11|
|2025-01-12|
|2025-01-14|
|2025-01-15|
|2025-01-16|
|2025-01-18|
|2025-01-19|
|2025-01-20|
|2025-01-21|
|2025-01-22|
|2025-01-23|
|2025-01-24|
+----------+
only showing top 20 rows


# 8. Basic Bronze → Silver preprocessing

The Silver layer follows the assignment rules and avoids aggressive cleansing. Exact duplicates are removed; missing text is labeled `Unknown`; valid numerical data is retained while clearly invalid values are nullified and then filled with column medians; obvious text casing/spacing issues are standardized; and dates are converted to one date type with invalid values left as null.

In [9]:
silver_df = bronze_df.dropDuplicates()

# ----- Text cleanup -----
string_columns = [name for name, dtype in silver_df.dtypes if dtype == "string"]

for column_name in string_columns:
    cleaned = F.trim(F.col(column_name))
    silver_df = silver_df.withColumn(
        column_name,
        F.when(F.length(cleaned) == 0, F.lit(None)).otherwise(cleaned)
    )

# Standardize only fields where simple capitalization is safe and useful.
for column_name in ["category", "city"]:
    silver_df = silver_df.withColumn(
        column_name,
        F.initcap(F.lower(F.trim(F.col(column_name))))
    )

# ----- Numeric cleanup -----
silver_df = silver_df.withColumn(
    "quantity",
    F.when(F.col("quantity").cast("double") > 0,
           F.col("quantity").cast("double"))
)

silver_df = silver_df.withColumn(
    "unit_price",
    F.when(F.col("unit_price").cast("double") > 0,
           F.col("unit_price").cast("double"))
)

silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(F.col("discount_percent").cast("double").between(0, 100),
           F.col("discount_percent").cast("double"))
)

# Fill numeric gaps with medians calculated from the cleaned numeric columns.
for column_name in ["quantity", "unit_price", "discount_percent"]:
    median_value = silver_df.approxQuantile(column_name, [0.5], 0.01)[0]
    silver_df = silver_df.fillna({column_name: float(median_value)})
    print(f"Median used for {column_name}: {median_value}")

# Fill missing text values after trimming/standardization.
if string_columns:
    silver_df = silver_df.fillna("Unknown", subset=string_columns)

# ----- Date cleanup -----
normalized_order_date = F.coalesce(*[
    F.try_to_timestamp(F.trim(F.col("order_date")), F.lit(pattern))
    for pattern in date_formats
])

silver_df = silver_df.withColumn(
    "order_date",
    normalized_order_date.cast("date")
)

# Keep a predictable column order: same columns as Bronze.
silver_df = silver_df.select(*bronze_df.columns)

print("Silver preprocessing completed.")

Median used for quantity: 5.0
Median used for unit_price: 1810.0
Median used for discount_percent: 20.0
Silver preprocessing completed.


In [10]:
# Verification before writing
silver_in_memory_count = silver_df.count()

print("=== Row Count Verification ===")
print(f"Before preprocessing: {bronze_count}")
print(f"After preprocessing:  {silver_in_memory_count}")
print(f"Rows removed by exact-duplicate step: {bronze_count - silver_in_memory_count}")

print("\n=== Sample of Silver Data ===")
silver_df.show(5, truncate=False)

print("\n=== Silver Schema ===")
silver_df.printSchema()

=== Row Count Verification ===
Before preprocessing: 1000
After preprocessing:  990
Rows removed by exact-duplicate step: 10

=== Sample of Silver Data ===
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4.0     |670.0     |0.0             |Net Banking     |Pune      |M

## 9. Save the Silver layer

Spark normally writes several `part-*.csv` files. For the assignment submission, this notebook coalesces the result to one CSV file and names it `sales_silver.csv`. A copy is also placed at `silver/sales_silver.csv`, matching the class GitHub submission structure.

In [11]:
# Remove previous Silver output so the notebook is safe to rerun.
if SILVER_DIR.exists():
    shutil.rmtree(SILVER_DIR)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

TEMP_OUTPUT = BASE / "silver" / "_spark_csv_tmp"
if TEMP_OUTPUT.exists():
    shutil.rmtree(TEMP_OUTPUT)

# Write one CSV partition, then copy it to the requested final filename.
(
    silver_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(TEMP_OUTPUT))
)

part_file = next(TEMP_OUTPUT.glob("part-*.csv"), None)
if part_file is None:
    raise FileNotFoundError("Spark did not create the expected CSV part file.")

shutil.copy2(part_file, SILVER_FILE)
shutil.copy2(part_file, SUBMISSION_SILVER)
shutil.rmtree(TEMP_OUTPUT)

print(f"Silver Data Lake file: {SILVER_FILE}")
print(f"GitHub submission copy: {SUBMISSION_SILVER}")

print("\nSilver folder contents:")
for item in sorted(SILVER_DIR.iterdir()):
    print(" -", item.name)

print("\nSubmission silver folder contents:")
for item in sorted(SUBMISSION_SILVER.parent.iterdir()):
    print(" -", item.name)

Silver Data Lake file: data_lake/silver/preprocessed_sales/sales_silver.csv
GitHub submission copy: silver/sales_silver.csv

Silver folder contents:
 - sales_silver.csv

Submission silver folder contents:
 - sales_silver.csv


## 10. Read the saved Silver data back with Spark

This final check proves that the file was written successfully and can be read again as a Spark DataFrame.

In [12]:
silver_readback = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(SILVER_FILE))
)

print("=== Silver Read-back Count ===")
print(silver_readback.count())

print("\n=== Silver Read-back Sample ===")
silver_readback.show(5, truncate=False)

print("\n=== Saved Silver Schema ===")
silver_readback.printSchema()

=== Silver Read-back Count ===
990

=== Silver Read-back Sample ===
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive

## 11. Preprocessing summary

The workflow performed the following basic operations:

1. Copied the uploaded raw `sales.csv` byte-for-byte into Bronze.
2. Measured exact duplicates and removed them from the Silver DataFrame.
3. Detected null/blank values and filled missing text with `Unknown`.
4. Trimmed whitespace and standardized obvious `category` and `city` capitalization variants.
5. Validated `quantity`, `unit_price`, and `discount_percent`; clearly invalid values were converted to null and then replaced with medians so useful records were retained.
6. Parsed supported date formats into one Spark `date` type; clearly invalid dates were left as null rather than guessed.
7. Saved the resulting Silver dataset as `sales_silver.csv` and read it back with Spark for verification.

### Expected submission files
```text
submissions/<ROLL_NUMBER>_<NAME>/
├── Data_Lake_Assignment.ipynb
└── silver/
    └── sales_silver.csv
```

Do **not** modify the class `README.md`, `dataset/`, or any other student's submission folder.